# KoHRM-Text-1.4B Colab T4 Pretraining Checkpoint Probe

This notebook loads the latest public KoHRM-Text-1.4B checkpoint without `transformers`, so it avoids the Colab `torchvision::nms` / custom `HrmTextConfig` import failure.

Important: this is a **rolling pretraining checkpoint probe**, not a final SFT/chat benchmark. KoHRM is currently trained with HRM-Text style single-stage instruction pretraining. It has not yet gone through the later behavior SFT/LoRA/RL pass, so strict JSON-only, command-only, grounded summary, and code-correctness tasks should be interpreted as post-training readiness checks.

Training format:

```text
<|im_start|><condition_token>instruction<|im_end|>response<|box_end|>
```

The helper follows upstream `InferenceCheckpoint.tokenize_prompt()`: `<boq><condition_tokens><instruction><eoq>`, then stops generation on `<|box_end|>`. Use `direct` / `<|object_ref_start|>` for answer-only completions.

The notebook separates:

- pretraining-distribution probes: simple continuation/QA tasks that check whether the checkpoint is learning the data distribution.
- strict post-training probes: JSON-only, command-only, and code-only tasks that will usually need SFT/LoRA/RL before they become reliable.

## 1. Install Dependencies

The runtime intentionally does not import `transformers`. `tokenizers` is pinned below `0.23.1` to avoid conflicts with Colab images that already contain `transformers 5.x`.

In [ ]:
!pip -q install -U huggingface_hub hf_transfer safetensors
!pip -q install --force-reinstall -q "tokenizers>=0.22.0,<0.23.1"

## 2. Runtime Settings

`MAX_SEQ_LEN=512` is the safest T4 default. Raise it to `768` only if the first load leaves enough free VRAM.

In [ ]:
import os
import json
import gc
import importlib.util
import subprocess
import sys
from pathlib import Path

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

REPO_ID = "LLM-OS-Models/KoHRM-Text-1.4B"
REVISION = "main"
LOCAL_DIR = Path("/content/KoHRM-Text-1.4B")
HELPER_PATH = LOCAL_DIR / "kohrm_colab_generate.py"
MAX_SEQ_LEN = 512

STRICT_SETTINGS = {
    "max_seq_len": MAX_SEQ_LEN,
    "temperature": 0.0,
    "top_p": 1.0,
    "repetition_penalty": 1.20,
    "no_repeat_ngram_size": 4,
    "condition": "direct",
}

print("repo:", REPO_ID)
print("revision:", REVISION)
print("local_dir:", LOCAL_DIR)
print("max_seq_len:", MAX_SEQ_LEN)

## 3. Download Latest Public Checkpoint

The public model repo is expected to contain `config.json`, `tokenizer.json`, `model.safetensors`, and the model card. If the lightweight helper is not present in the model repo, the notebook clones the GitHub repo and copies the helper from `notebooks/`.

In [ ]:
from huggingface_hub import snapshot_download

LOCAL_DIR.mkdir(parents=True, exist_ok=True)
patterns = [
    "README.md",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "special_tokens_map.json",
    "model.safetensors",
    "kohrm_colab_generate.py",
    "notebooks/kohrm_colab_generate.py",
]

snapshot_download(
    repo_id=REPO_ID,
    repo_type="model",
    revision=REVISION,
    local_dir=str(LOCAL_DIR),
    local_dir_use_symlinks=False,
    allow_patterns=patterns,
)

nested_helper = LOCAL_DIR / "notebooks" / "kohrm_colab_generate.py"
if not HELPER_PATH.exists() and nested_helper.exists():
    HELPER_PATH.write_text(nested_helper.read_text(encoding="utf-8"), encoding="utf-8")

if not HELPER_PATH.exists():
    repo = Path("/content/KoHRM-text")
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/LLM-OS-Models/KoHRM-text", str(repo)],
            check=True,
        )
    HELPER_PATH.write_text((repo / "notebooks" / "kohrm_colab_generate.py").read_text(encoding="utf-8"), encoding="utf-8")

for name in ["config.json", "tokenizer.json", "model.safetensors", "README.md", "kohrm_colab_generate.py"]:
    path = LOCAL_DIR / name
    print(f"{name}: {path.exists()} {path.stat().st_size / 2**20:.2f} MiB" if path.exists() else f"{name}: missing")

## 4. Inspect Config and Prompt Format

Do not use `AutoTokenizer` or `AutoModelForCausalLM` here. The current public export uses a custom HRM-Text architecture, and the helper below loads it directly from `safetensors`.

In [ ]:
spec = importlib.util.spec_from_file_location("kohrm_colab_generate", HELPER_PATH)
kohrm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(kohrm)

config = json.loads((LOCAL_DIR / "config.json").read_text(encoding="utf-8"))
print(json.dumps({
    "model_type": config.get("model_type"),
    "architectures": config.get("architectures"),
    "vocab_size": config.get("vocab_size"),
    "hidden_size": config.get("hidden_size"),
    "num_hidden_layers": config.get("num_hidden_layers"),
    "num_attention_heads": config.get("num_attention_heads"),
    "H_cycles": config.get("H_cycles"),
    "L_cycles": config.get("L_cycles"),
    "max_position_embeddings": config.get("max_position_embeddings"),
    "prefix_lm": config.get("prefix_lm"),
}, indent=2, ensure_ascii=False))

from tokenizers import Tokenizer
raw_tok = Tokenizer.from_file(str(LOCAL_DIR / "tokenizer.json"))
for token in ["<|im_start|>", "<|object_ref_start|>", "<|object_ref_end|>", "<|quad_start|>", "<|quad_end|>", "<|im_end|>", "<|box_end|>"]:
    print(f"{token:22s}", raw_tok.token_to_id(token))

print("direct condition token:", kohrm.condition_to_tokens("direct"))
example = kohrm.format_kohrm_prompt("Return one bash command only.", condition="direct")
print("wrapped prompt:", example)

## 5. Load Model Once

On a T4, loading can take a few minutes. The helper uses PyTorch scaled-dot-product attention and a static KV cache. It is slower than the training-time FlashAttention path, but it is portable enough for Colab smoke tests.

In [ ]:
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(torch.cuda.get_device_name(0))

model, tokenizer, cfg = kohrm.load_kohrm(LOCAL_DIR, max_gpu_memory_gib=14.0)
print("loaded dtype:", next(model.parameters()).dtype)
print("loaded device:", next(model.parameters()).device)
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory free/total GiB after load: {free / 2**30:.2f}/{total / 2**30:.2f}")

## 6. Probe Cases

The first block is intentionally easier and closer to pretraining distribution. The second block is stricter and should be read as a post-training target, not as a final failure verdict for the current unsupervised/SFT-free checkpoint.

Prompts here are deliberately plain. Over-complicated meta-prompts are not a good fix for a pretraining checkpoint; behavior such as JSON-only, command-only, and exact code generation belongs in the later SFT/LoRA/RL pass.

In [ ]:
import ast
import json
import re

PRETRAINING_PROBES = [
    {
        "name": "ko_finance_plain_qa",
        "phase": "pretraining_probe",
        "lang": "ko",
        "expect": "ko_contains_terms",
        "required_terms": ["환율", "투자"],
        "max_new_tokens": 96,
        "prompt": "환율 변동이 개인 투자에 미치는 영향을 간단히 설명하세요.",
    },
    {
        "name": "ko_legal_plain_extraction",
        "phase": "pretraining_probe",
        "lang": "ko",
        "expect": "ko_contains_terms",
        "required_terms": ["홍보대사", "무보수"],
        "max_new_tokens": 96,
        "prompt": """다음 조문에서 핵심 내용을 한 문장으로 말하세요.

제5조 (보상)
① 홍보대사는 무보수 명예직으로 한다.
② 군수는 홍보대사가 임무 수행을 위하여 활동하는 경우 예산의 범위 안에서 홍보활동에 직접 소요되는 실 경비로 숙식비, 차량운행 경비, 기타 비용과 격려금품을 지급할 수 있다.""",
    },
    {
        "name": "ko_wiki_plain_qa",
        "phase": "pretraining_probe",
        "lang": "ko",
        "expect": "ko_contains_terms",
        "required_terms": ["훈민정음", "세종"],
        "max_new_tokens": 96,
        "prompt": "훈민정음은 누가 만들었고 어떤 목적이 있었나요?",
    },
    {
        "name": "en_terminal_intent_completion",
        "phase": "pretraining_probe",
        "lang": "en",
        "expect": "mentions_shell_concept",
        "required_terms": ["find", "sort"],
        "max_new_tokens": 96,
        "prompt": "In bash, to list the largest files under the current directory, you can use",
    },
]

STRICT_POSTTRAINING_PROBES = [
    {
        "name": "ko_legal_json_direct",
        "phase": "posttraining_strict_probe",
        "lang": "ko",
        "expect": "strict_json_keys",
        "required_keys": ["조문명", "적용 대상", "핵심 의무"],
        "required_terms": ["제5조", "홍보대사", "무보수", "실 경비"],
        "max_new_tokens": 128,
        "prompt": """다음 한국 법령/행정규칙 발췌문에서 조문명, 적용 대상, 핵심 의무를 JSON 객체 하나로만 추출하세요. JSON 밖의 설명은 쓰지 마세요.

[문서명]
진안군 홍보대사 운영 조례

[조문]
제5조 (보상)
① 홍보대사는 무보수 명예직으로 한다.
② 군수는 홍보대사가 임무 수행을 위하여 활동하는 경우 예산의 범위 안에서 홍보활동에 직접 소요되는 실 경비로 숙식비, 차량운행 경비, 기타 비용과 격려금품을 지급할 수 있다.""",
    },
    {
        "name": "ko_wiki_grounded_summary",
        "phase": "posttraining_strict_probe",
        "lang": "ko",
        "expect": "ko_grounded_summary",
        "required_terms": ["훈민정음", "세종"],
        "forbidden_terms": ["ganjang", "A:", "<br>", "</br>"],
        "max_new_tokens": 96,
        "prompt": """다음 글만 근거로 핵심 내용을 한국어로 3문장 이내로 요약하세요. 글에 없는 사실은 추가하지 마세요.

[글]
훈민정음은 조선 세종이 창제한 문자 체계이다. 창제 목적은 백성이 자신의 뜻을 쉽게 글로 표현하도록 돕는 데 있었다. 자음은 발음 기관의 모양을 본떠 만들었고, 모음은 하늘, 땅, 사람의 원리를 바탕으로 구성되었다.""",
    },
    {
        "name": "ko_finance_short",
        "phase": "posttraining_strict_probe",
        "lang": "ko",
        "expect": "ko_grounded_summary",
        "required_terms": ["환율", "투자"],
        "forbidden_terms": ["<br>", "</br>", "A:", "B:", "C:"],
        "max_new_tokens": 96,
        "prompt": "환율 변동이 개인 투자에 미치는 영향과 대비 전략을 한국어로 4문장 이내로 설명하세요. 같은 표현을 반복하지 마세요.",
    },
    {
        "name": "en_terminal_command_only",
        "phase": "posttraining_strict_probe",
        "lang": "en",
        "expect": "strict_shell_command",
        "required_terms": ["find", "sort", "head"],
        "max_new_tokens": 64,
        "prompt": "Return one bash command only. No explanation. Task: find the 10 largest files under the current directory, excluding .git, sorted by size descending.",
    },
    {
        "name": "en_tool_call_json",
        "phase": "posttraining_strict_probe",
        "lang": "en",
        "expect": "strict_json_keys",
        "required_keys": ["tool", "args"],
        "required_terms": ["shell", "du"],
        "max_new_tokens": 96,
        "prompt": "Return one JSON object only for a terminal tool call. Schema: {\"tool\": \"shell\", \"args\": {\"cmd\": string}}. Task: print current disk usage for the current directory in human-readable form.",
    },
    {
        "name": "en_python_code_only",
        "phase": "posttraining_strict_probe",
        "lang": "en",
        "expect": "strict_python_function",
        "required_terms": ["top_k_lengths"],
        "max_new_tokens": 128,
        "prompt": "Write Python code only. Define a function top_k_lengths(items, k) that returns the k longest strings from items, preserving original order for ties.",
    },
]

TEST_CASES = PRETRAINING_PROBES + STRICT_POSTTRAINING_PROBES


def strip_markdown_fence(text):
    stripped = text.strip()
    if stripped.startswith("```"):
        stripped = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", stripped)
        stripped = re.sub(r"\s*```$", "", stripped)
    return stripped.strip()


def has_forbidden(text, case):
    return [term for term in case.get("forbidden_terms", []) if term in text]


def missing_terms(text, case):
    return [term for term in case.get("required_terms", []) if term not in text]


def validate_output(case, text):
    expect = case["expect"]
    raw = text.strip()
    body = strip_markdown_fence(raw)
    if not raw:
        return "FAIL: empty output"

    forbidden = has_forbidden(raw, case)
    if forbidden:
        return f"FAIL: forbidden artifacts {forbidden}"

    if expect == "ko_contains_terms":
        missing = missing_terms(raw, case)
        if missing:
            return f"WARN: pretraining probe missing expected terms {missing}"
        if len(re.findall(r"[가-힣]", raw)) < 5:
            return "WARN: too little Korean text"
        return "PASS: pretraining distribution signal"

    if expect == "mentions_shell_concept":
        missing = missing_terms(raw, case)
        if missing:
            return f"WARN: shell concept missing terms {missing}"
        return "PASS: mentions expected shell concepts"

    if expect == "strict_json_keys":
        if raw != body:
            return "FAIL: JSON wrapped in markdown fence; SFT/format tuning needed"
        try:
            obj = json.loads(body)
        except Exception as exc:
            return f"FAIL: invalid JSON ({exc.__class__.__name__})"
        if not isinstance(obj, dict):
            return "FAIL: JSON is not an object"
        missing_keys = [key for key in case.get("required_keys", []) if key not in obj]
        if missing_keys:
            return f"FAIL: JSON missing keys {missing_keys}"
        missing = missing_terms(json.dumps(obj, ensure_ascii=False), case)
        if missing:
            return f"FAIL: JSON content missing terms {missing}"
        return "PASS: strict JSON"

    if expect == "ko_grounded_summary":
        missing = missing_terms(raw, case)
        if missing:
            return f"FAIL: missing grounded Korean terms {missing}"
        if len(re.findall(r"[가-힣]", raw)) < 10:
            return "FAIL: too little Korean text"
        if len(raw.splitlines()) > 4:
            return "WARN: too many lines for short summary"
        return "PASS: Korean grounded-form probe"

    if expect == "strict_shell_command":
        if "\n" in raw:
            return "FAIL: more than one line"
        if any(marker in raw for marker in ["```", "We need", "Step", "1.", "Task:"]):
            return "FAIL: contains explanation/formatting"
        if not re.match(r"^(find|du|ls|python|sh|bash|fd|rg)\b", raw):
            return "FAIL: not a shell command start"
        missing = missing_terms(raw, case)
        if missing:
            return f"FAIL: command missing expected terms {missing}"
        return "PASS: strict shell command"

    if expect == "strict_python_function":
        if "```" in raw:
            return "FAIL: markdown fence present"
        try:
            tree = ast.parse(raw)
        except SyntaxError as exc:
            return f"FAIL: Python syntax error line {exc.lineno}"
        funcs = [node for node in tree.body if isinstance(node, ast.FunctionDef)]
        if not funcs:
            return "FAIL: no function definition"
        if funcs[0].name != "top_k_lengths":
            return f"FAIL: wrong function name {funcs[0].name!r}"
        return "PASS: syntactically valid target function"

    return "WARN: unchecked expectation"


results = []
for case in TEST_CASES:
    print("=" * 80)
    print("case:", case["name"])
    print("phase:", case["phase"])
    print("expect:", case["expect"])
    print("prompt:", case["prompt"])
    output = kohrm.generate_from_loaded(
        model,
        tokenizer,
        cfg,
        case["prompt"],
        max_new_tokens=case["max_new_tokens"],
        **STRICT_SETTINGS,
    )
    verdict = validate_output(case, output)
    results.append({"case": case["name"], "phase": case["phase"], "expect": case["expect"], "verdict": verdict, "output": output})
    print("verdict:", verdict)
    print("--- output ---")
    print(output)

print("=" * 80)
print(json.dumps([{k: r[k] for k in ["case", "phase", "expect", "verdict"]} for r in results], indent=2, ensure_ascii=False))

## 7. Optional Low-Temperature Retry

This retry is only diagnostic. It should not turn a strict post-training failure into a pass. Use it to see whether the bad output is a greedy decoding artifact or a checkpoint/format-alignment issue.

In [ ]:
RETRY_CASE_NAMES = {"ko_finance_plain_qa", "ko_finance_short", "en_terminal_command_only"}
RETRY_SETTINGS = dict(STRICT_SETTINGS)
RETRY_SETTINGS.update({
    "temperature": 0.2,
    "top_p": 0.85,
    "repetition_penalty": 1.22,
    "no_repeat_ngram_size": 4,
})

for case in TEST_CASES:
    if case["name"] not in RETRY_CASE_NAMES:
        continue
    print("=" * 80)
    print("retry case:", case["name"])
    output = kohrm.generate_from_loaded(
        model,
        tokenizer,
        cfg,
        case["prompt"],
        max_new_tokens=case["max_new_tokens"],
        **RETRY_SETTINGS,
    )
    print("retry verdict:", validate_output(case, output))
    print(output)

## 8. Interpretation Checklist

Do not read this notebook as a final KoHRM chat/SFT benchmark. The current public checkpoint is a rolling pretraining checkpoint. It has not yet received the later behavior SFT/LoRA/RL pass.

Use the results this way:

- A pretraining probe `PASS` means the checkpoint shows a usable signal from that domain or task distribution.
- A pretraining probe `WARN` means the signal is weak or the prompt is still outside the learned distribution.
- A strict post-training probe `FAIL` is expected at this stage if the model has not learned stable JSON-only, command-only, grounded summary, or code-only behavior.
- A strict post-training probe `PASS` is useful, but it is not required before SFT.

For strict behavior, compare the same probes after the final continuation checkpoint and after `behavior_mini` / `terminal_tool_core` / `korean_domain_core` LoRA. The validator intentionally marks markdown-wrapped JSON, non-command text, syntactically invalid code, and non-Korean placeholder text as failures.